# Pipelines DVC et compression de modèles

Nous allons construire, avec DVC, la chaîne de traitements d'un classifieur d'images de paysages : préparation des données, entraînement, puis compression du modèle.

Le dépôt [shuuchuu/compression](https://github.com/shuuchuu/compression) porte le même énoncé, pour faire ces travaux pratiques dans un Codespace GitHub ou un clone local, et leur solution sur sa branche `solution`.

Les quatre premières étapes (le paquet et le dépôt DVC, l'ajout des données, l'étape `prepare`, le stockage distant) reprennent les gestes de base de DVC (`dvc init`, `dvc add`, `dvc stage add`, `dvc repro`, `dvc push`), pratiqués dans les travaux pratiques « Versionnage et traçage des données » : leurs cellules sont fournies, avec un rappel de ce qu'elles font. Les exercices commencent à l'entraînement : les paramètres et la métrique d'une étape, les expériences, puis la compression du modèle.

In [ ]:
# @title Mise en place de l'environnement
!rm -rf sample_data .config
!git config --global user.email "jane@doe.eu"
!git config --global user.name "Jane Doe"
!git config --global init.defaultBranch main
!git init
!echo "__pycache__/" > .gitignore
!pip install -U uv

## Création du paquet Python et du dépôt DVC

`uv init` crée le paquet Python `compression_modele` (pas `compression` : la bibliothèque standard de Python 3.14 a un paquet de ce nom, qui masquerait le nôtre), ses sources dans `src/compression_modele/`, en Python 3.13, que `--python` fixe : sans lui, `uv init` prend le Python le plus récent installé, et onnxruntime n'a pas de roues pour Python 3.15.

In [ ]:
!uv init --package --name compression_modele --python 3.13 .

La cellule suivante ajoute à `pyproject.toml`, avant toute installation, l'index des roues CPU de PyTorch, réservé (`explicit = true`) aux paquets que `[tool.uv.sources]` lui attribue : les roues de PyPI embarquent les bibliothèques CUDA, plusieurs gigaoctets inutiles ici, où tout tourne sur CPU.

In [ ]:
%%writefile -a pyproject.toml

[[tool.uv.index]]
name = "pytorch-cpu"
url = "https://download.pytorch.org/whl/cpu"
explicit = true

[tool.uv.sources]
torch = { index = "pytorch-cpu" }
torchvision = { index = "pytorch-cpu" }

`uv add` installe DVC et les bibliothèques du code fourni plus loin (`onnxscript` sert à l'export ONNX de PyTorch) dans l'environnement du projet, `.venv`, où `uv run` lance DVC ; `dvc init` crée le dépôt DVC, `.dvc/`, et ajoute lui-même ses fichiers à l'index de git.

In [ ]:
!uv add dvc onnx onnxruntime onnxscript torch torchvision
!uv run dvc init
!git add .
!git commit -m "Création du paquet Python et du dépôt DVC"

## Ajout des données à DVC

La cellule suivante télécharge dans `donnees/` le dossier `landscape` du dépôt [shuuchuu/datasets](https://github.com/shuuchuu/datasets/tree/main/landscape), sans `donnees/seg_pred/`, des images sans étiquette qui ne servent pas ici. `donnees/seg_train/` et `donnees/seg_test/` contiennent les images d'entraînement et de test (150 × 150 pixels), un dossier par classe (`buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`).

In [ ]:
# @title Téléchargement des données
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!mv landscape donnees
!rm -r donnees/seg_pred

`dvc add` met `donnees/` dans le cache de DVC, `.dvc/cache`, et écrit `donnees.dvc`, qui retient son empreinte md5 : git suit ce fichier, et `.gitignore`, où DVC ajoute `/donnees`, pas les images.

In [ ]:
!uv run dvc add donnees
!git add .gitignore donnees.dvc
!git commit -m "Ajout des données"

## Préparation des données

L'entraînement et la compression lisent les mêmes images. Plutôt que de les décoder et de les redimensionner à chaque fois, une première étape de la chaîne, `prepare`, le fait une fois pour toutes : elle copie les `images_par_classe` premières images de chaque classe dans `preparees/entrainement/` et `preparees/test/`, réduites à `TAILLE_IMAGE`, 88 × 88 pixels. 500 images par classe suffisent à ces travaux pratiques, et un entraînement sur CPU dure alors une à deux minutes. À cette taille, les dernières cartes de caractéristiques du LeNet de l'entraînement font 20 × 20, que sa couche `AdaptiveAvgPool2d` réduit à 5 × 5 par des moyennes sur des fenêtres régulières, ce que l'export ONNX traduit en une seule opération (à 100 × 100, il en faut une centaine).

Son code, dans `src/compression_modele/donnees.py` (la cellule suivante l'écrit), transforme aussi les images en tenseurs, leurs pixels mis à l'échelle [0, 1] (`cree_transformation`), et les charge par lots de 64 (`cree_chargeur`) :

In [ ]:
%%writefile src/compression_modele/donnees.py
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

TAILLE_IMAGE = (88, 88)


def cree_transformation(taille_image: tuple[int, int]) -> v2.Compose:
  return v2.Compose(
    [v2.ToImage(), v2.Resize(taille_image), v2.ToDtype(torch.float32, scale=True)]
  )


def cree_chargeur(
  dossier: str, taille_image: tuple[int, int], melange: bool = False
) -> DataLoader:
  jeu = datasets.ImageFolder(dossier, transform=cree_transformation(taille_image))
  return DataLoader(jeu, batch_size=64, shuffle=melange)


def prepare(
  dossier_ent: str, dossier_test: str, images_par_classe: int, dossier_prepare: str
) -> None:
  for partie, dossier in (("entrainement", dossier_ent), ("test", dossier_test)):
    for dossier_classe in sorted(Path(dossier).iterdir()):
      dossier_sortie = Path(dossier_prepare) / partie / dossier_classe.name
      dossier_sortie.mkdir(parents=True, exist_ok=True)
      for chemin in sorted(dossier_classe.iterdir())[:images_par_classe]:
        image = Image.open(chemin).convert("RGB").resize(TAILLE_IMAGE)
        image.save(dossier_sortie / chemin.name, quality=95)

La suivante écrit le point d'entrée de la chaîne, `src/compression_modele/__main__.py` : `python -m compression_modele <étape>` lance la fonction de cette étape, `prepare` de `donnees.py`, puis `entraine` de `entrainement.py` et `compresse` de `compression.py`, que vous écrirez plus loin, avec les paramètres que `dvc.yaml` lui déclare, lus dans `params.yaml` par `dvc.api.params_show`. Le module d'une étape n'est importé que quand elle s'exécute : ce fichier ne changera plus.

In [ ]:
%%writefile src/compression_modele/__main__.py
from importlib import import_module
from sys import argv

from dvc.api import params_show

MODULES = {"prepare": "donnees", "entraine": "entrainement", "compresse": "compression"}

if len(argv) != 2 or argv[1] not in MODULES:
  message = f"Usage : python -m compression_modele {{{','.join(MODULES)}}}"
  raise SystemExit(message)

etape = argv[1]
fonction = getattr(import_module(f"compression_modele.{MODULES[etape]}"), etape)
fonction(**params_show(stages=[etape]))

Les deux cellules suivantes ajoutent l'étape `prepare` à la chaîne : ses paramètres dans `params.yaml`, puis `dvc stage add`, qui l'écrit dans `dvc.yaml` avec son nom (`-n`), ses paramètres (`-p`), ses dépendances (`-d` : les images et le code qu'elle exécute) et sa sortie (`-o`). `dvc repro` l'exécute et retient dans `dvc.lock` les empreintes de ce qu'elle a lu et produit ; DVC ajoute `/preparees` à `.gitignore`.

In [ ]:
%%writefile params.yaml
dossier_ent: donnees/seg_train
dossier_test: donnees/seg_test
images_par_classe: 500
dossier_prepare: preparees

In [ ]:
!uv run dvc stage add -n prepare -p dossier_ent,dossier_test,images_par_classe,dossier_prepare -d donnees/seg_train -d donnees/seg_test -d src/compression_modele/__main__.py -d src/compression_modele/donnees.py -o preparees python -m compression_modele prepare
!uv run dvc repro
!git add .gitignore dvc.lock dvc.yaml params.yaml src
!git commit -m "Ajout de l'étape de préparation"

## Stockage distant

Le cache de DVC n'existe que sur cette machine : `dvc push` l'envoie au stockage distant que `dvc remote add -d` déclare par défaut dans `.dvc/config`, ici un simple dossier, `/tmp/stockage-dvc`. Sur une autre machine, `git clone` puis `dvc pull` retrouveraient `donnees/` et `preparees/` sans rien recalculer, si le stockage était partagé (un compartiment S3, un serveur SSH…).

In [ ]:
!uv run dvc remote add -d local /tmp/stockage-dvc
!git add .dvc/config
!git commit -m "Ajout du stockage distant"
!uv run dvc push

## Entraînement d'un modèle simple

Nous allons maintenant entraîner un modèle de classification d'images : le modèle historique LeNet, deux convolutions puis trois couches denses. La cellule suivante écrit son code dans `src/compression_modele/modele.py` :

In [ ]:
%%writefile src/compression_modele/modele.py
import torch
from torch import nn


class LeNet(nn.Module):
  def __init__(self, nb_classes: int = 6) -> None:
    super().__init__()
    self.caracteristiques = nn.Sequential(
      nn.Conv2d(3, 6, kernel_size=5, padding=2),
      nn.ReLU(),
      nn.MaxPool2d(2),
      nn.Conv2d(6, 16, kernel_size=5),
      nn.ReLU(),
      nn.MaxPool2d(2),
      nn.AdaptiveAvgPool2d((5, 5)),
    )
    self.classifieur = nn.Sequential(
      nn.Flatten(),
      nn.Linear(16 * 5 * 5, 120),
      nn.ReLU(),
      nn.Linear(120, 84),
      nn.ReLU(),
      nn.Linear(84, nb_classes),
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.classifieur(self.caracteristiques(x))

Celle-ci écrit celui de l'entraînement, dans `src/compression_modele/entrainement.py` : `entraine_epoque` passe une fois sur les images d'entraînement, `evalue` mesure l'exactitude (*accuracy*, la part d'images bien classées), et `entraine` enchaîne `nb_epoques` époques, puis enregistre les poids du modèle dans `chemin_modele` et son exactitude sur les images de test dans `metriques/entrainement.json`, un fichier de métriques que DVC sait lire et comparer :

In [ ]:
%%writefile src/compression_modele/entrainement.py
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from .donnees import TAILLE_IMAGE, cree_chargeur
from .modele import LeNet

CHEMIN_METRIQUES = Path("metriques/entrainement.json")


def entraine_epoque(
  modele: nn.Module, chargeur: DataLoader, optimiseur: torch.optim.Optimizer
) -> float:
  critere = nn.CrossEntropyLoss()
  modele.train()
  total, nombre = 0.0, 0
  for images, etiquettes in chargeur:
    optimiseur.zero_grad()
    perte = critere(modele(images), etiquettes)
    perte.backward()
    optimiseur.step()
    total += perte.item() * len(etiquettes)
    nombre += len(etiquettes)
  return total / nombre


def evalue(modele: nn.Module, chargeur: DataLoader) -> float:
  modele.eval()
  corrects, nombre = 0, 0
  with torch.inference_mode():
    for images, etiquettes in chargeur:
      corrects += int((modele(images).argmax(dim=1) == etiquettes).sum())
      nombre += len(etiquettes)
  return corrects / nombre


def entraine(
  dossier_prepare: str, taux_apprentissage: float, nb_epoques: int, chemin_modele: str
) -> None:
  torch.manual_seed(0)
  chargeur_ent = cree_chargeur(
    f"{dossier_prepare}/entrainement", TAILLE_IMAGE, melange=True
  )
  chargeur_test = cree_chargeur(f"{dossier_prepare}/test", TAILLE_IMAGE)
  modele = LeNet()
  optimiseur = torch.optim.AdamW(modele.parameters(), lr=taux_apprentissage)
  for epoque in range(nb_epoques):
    perte = entraine_epoque(modele, chargeur_ent, optimiseur)
    print(f"époque {epoque + 1}/{nb_epoques} : perte={perte:.3f}")
  exactitude = evalue(modele, chargeur_test)
  print(f"exactitude={exactitude:.3f}")
  torch.save(modele.state_dict(), chemin_modele)
  CHEMIN_METRIQUES.parent.mkdir(exist_ok=True)
  CHEMIN_METRIQUES.write_text(json.dumps({"exactitude": exactitude, "perte": perte}))

Sur le modèle de l'étape `prepare`, ajoutez à la chaîne l'étape `entraine`, qui lance `python -m compression_modele entraine` : ses paramètres `taux_apprentissage` (`0.001`), `nb_epoques` (`5`) et `chemin_modele` (`classifieur_paysages.pt`) s'ajoutent à `params.yaml` (`%%writefile -a params.yaml` ajoute à la fin du fichier), et elle lit aussi `dossier_prepare`. Déclarez `metriques/entrainement.json` comme métrique de l'étape, exécutez la chaîne, affichez les métriques avec `dvc metrics show`, puis enregistrez le tout dans git.

**Documentation utile**

- [`dvc stage add`](https://doc.dvc.org/command-reference/stage/add)
- [`dvc repro`](https://doc.dvc.org/command-reference/repro)
- [`dvc metrics show`](https://doc.dvc.org/command-reference/metrics/show)

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

### Conseils supplémentaires

Les options de `dvc stage add`, comme pour `prepare` : `-n` nomme l'étape, `-p` déclare ses paramètres (leurs noms dans `params.yaml`, séparés par des virgules), `-d` chaque dépendance, `-o` sa sortie, puis vient la commande. Les dépendances de `entraine` sont la sortie de `prepare` et le code qu'elle exécute (`__main__.py`, `donnees.py`, `modele.py` et `entrainement.py`) : c'est en déclarant `preparees` comme dépendance que DVC sait que `entraine` vient après `prepare`. Sa sortie est le modèle (`-o`). Sa métrique se déclare avec `-M` : un fichier que DVC ne met pas en cache et que git suit, pour comparer les métriques d'un commit à l'autre. `dvc repro` ne relance pas `prepare`, qui est à jour. L'entraînement dure une à deux minutes.

```
%%writefile -a params.yaml
taux_apprentissage: ...
nb_epoques: ...
chemin_modele: ...
```

```python
!uv run dvc stage add -n entraine -p ... -d preparees -d ... -o ... -M ... python -m compression_modele entraine
!uv run dvc repro
!uv run dvc metrics show
!git add ...
!git commit -m "..."
```

### Solution

In [ ]:
%%writefile -a params.yaml
taux_apprentissage: 0.001
nb_epoques: 5
chemin_modele: classifieur_paysages.pt

In [ ]:
!uv run dvc stage add -n entraine -p dossier_prepare,taux_apprentissage,nb_epoques,chemin_modele -d preparees -d src/compression_modele/__main__.py -d src/compression_modele/donnees.py -d src/compression_modele/modele.py -d src/compression_modele/entrainement.py -o classifieur_paysages.pt -M metriques/entrainement.json python -m compression_modele entraine

In [ ]:
!uv run dvc repro

In [ ]:
!uv run dvc metrics show

In [ ]:
!git add .gitignore dvc.lock dvc.yaml metriques params.yaml src

In [ ]:
!git commit -m "Ajout de l'étape d'entraînement"

### Une expérience : changement du taux d'apprentissage

Lancez avec `dvc exp run` une expérience où `taux_apprentissage` vaut `0.003`, sans modifier `params.yaml` à la main. Comparez ses métriques à celles du dernier commit avec `dvc metrics diff`, puis listez les expériences avec `dvc exp show`. Quelles étapes l'expérience a-t-elle exécutées ? Lesquelles s'exécuteraient si vous changiez `images_par_classe` ? Si l'exactitude s'améliore, gardez ce taux : enregistrez dans git ce que l'expérience a changé.

**Documentation utile**

- [`dvc exp run`](https://doc.dvc.org/command-reference/exp/run)
- [`dvc metrics diff`](https://doc.dvc.org/command-reference/metrics/diff)
- [`dvc exp show`](https://doc.dvc.org/command-reference/exp/show)

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

#### Conseils supplémentaires

`-S taux_apprentissage=0.003` change le paramètre le temps de l'expérience, et `dvc exp run` en laisse le résultat dans l'espace de travail : `params.yaml`, `dvc.lock`, le modèle et `metriques/entrainement.json`. `dvc metrics diff` compare l'espace de travail au dernier commit ; `--only-changed` n'affiche, dans `dvc exp show`, que les colonnes qui diffèrent d'une expérience à l'autre. Pour `images_par_classe`, regardez de quoi dépend `entraine`.

```python
!uv run dvc exp run -S ...
!uv run dvc metrics diff
!uv run dvc exp show --only-changed
!git add ...
!git commit -m "..."
```

#### Solution

In [ ]:
!uv run dvc exp run -S taux_apprentissage=0.003

In [ ]:
!uv run dvc metrics diff

In [ ]:
!uv run dvc exp show --only-changed

In [ ]:
!git add dvc.lock metriques params.yaml

In [ ]:
!git commit -m "Taux d'apprentissage à 0,003"

Seule `entraine` s'exécute : `prepare` ne lit pas `taux_apprentissage`. Sur nos essais, l'exactitude passe de 0,58 à 0,61 environ (elle varie un peu d'une machine à l'autre). Changer `images_par_classe` rendrait `prepare` obsolète, et `entraine` s'exécuterait après elle, parce que sa dépendance `preparees` aurait changé. Si `prepare` produisait exactement les mêmes fichiers, DVC ne relancerait pas `entraine` : il compare les empreintes, pas les dates. `dvc exp show` garde la trace de chaque expérience, même non commitée : `dvc exp apply` en ramène une dans l'espace de travail, `dvc exp branch` en fait une branche.

## Compression du modèle a posteriori

Reste l'étape `compresse`, qui réduit le modèle entraîné et mesure ce que la réduction coûte et rapporte, selon deux paramètres :

- `parcimonie` : la part des poids des convolutions et des couches denses à mettre à zéro, par un élagage non structuré, global, des poids de plus faible magnitude, suivi d'une époque de réentraînement ; `0` n'élague rien ;
- `quantification` : `aucune` (pas de quantification), `dynamique` ou `statique`, les quantifications post-entraînement en int8 d'ONNX Runtime.

Complétez la fonction `compresse` de `src/compression_modele/compression.py`, dans la cellule suivante, dont le début fournit :

- `exporte_onnx`, qui exporte un modèle au format ONNX, en un seul fichier, son entrée nommée `x` ;
- `Calibration`, qui donne à `quantize_static` des images une à une ;
- `mesure`, qui mesure l'exactitude d'un modèle ONNX sur des images, la taille de son fichier, brute (`taille_mo`) et compressée par gzip (`gzip_mo`), et sa latence pour une image, médiane (`latence_ms`) et 95ᵉ centile (`p95_ms`), sans les 10 premières mesures (la chauffe) ;
- `rassemble_images`, qui rassemble les images et les étiquettes d'un `DataLoader` en tableaux.

La fonction doit :

1. charger le LeNet entraîné (`chemin_modele`) et l'exporter en ONNX : c'est le modèle flottant de référence ;
2. si `parcimonie` est positif, élaguer le modèle, le réentraîner une époque avec `entraine_epoque` et un optimiseur `AdamW` de taux `taux_apprentissage`, puis rendre l'élagage définitif ;
3. exporter le modèle élagué, puis le quantifier selon `quantification` dans `chemin_onnx` (`aucune` : une simple copie), en calibrant la quantification statique sur un lot d'images d'entraînement ;
4. écrire dans `CHEMIN_METRIQUES`, en JSON, les mesures du modèle flottant et du modèle réduit sur les images de test, sous les clés `flottant` et `compresse`.

**Documentation utile**

- [`torch.load`](https://docs.pytorch.org/docs/stable/generated/torch.load.html)
- [`torch.nn.Module.load_state_dict`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.load_state_dict)
- [`tempfile.TemporaryDirectory`](https://docs.python.org/3/library/tempfile.html#tempfile.TemporaryDirectory)
- [`shutil.copyfile`](https://docs.python.org/3/library/shutil.html#shutil.copyfile)

In [ ]:
%%writefile src/compression_modele/compression.py
import gzip
import json
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from onnxruntime.quantization import (
  CalibrationDataReader,
  QuantType,
  quantize_dynamic,
  quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader

from .donnees import TAILLE_IMAGE, cree_chargeur
from .entrainement import entraine_epoque
from .modele import LeNet

CHEMIN_METRIQUES = Path("metriques/compression.json")


def exporte_onnx(modele: nn.Module, chemin: str) -> None:
  exemple = (torch.rand(1, 3, *TAILLE_IMAGE),)
  torch.onnx.export(
    modele.eval(), exemple, chemin, dynamo=True, external_data=False, verbose=False
  )


class Calibration(CalibrationDataReader):
  def __init__(self, images: list[torch.Tensor]) -> None:
    self.images = iter(images)

  def get_next(self) -> dict | None:
    image = next(self.images, None)
    return None if image is None else {"x": image.numpy()}


def mesure(chemin: str, images: np.ndarray, etiquettes: np.ndarray) -> dict:
  session = ort.InferenceSession(chemin, providers=["CPUExecutionProvider"])
  predictions, durees = [], []
  for image in images:
    debut = time.perf_counter()
    (logits,) = session.run(None, {"x": image[None]})
    durees.append(time.perf_counter() - debut)
    predictions.append(logits.argmax())
  return {
    "exactitude": float(np.mean(np.array(predictions) == etiquettes)),
    "taille_mo": Path(chemin).stat().st_size / 1e6,
    "gzip_mo": len(gzip.compress(Path(chemin).read_bytes())) / 1e6,
    "latence_ms": float(1000 * np.median(durees[10:])),
    "p95_ms": float(1000 * np.percentile(durees[10:], 95)),
  }


def rassemble_images(chargeur: DataLoader) -> tuple[np.ndarray, np.ndarray]:
  images, etiquettes = zip(*chargeur, strict=True)
  return torch.cat(images).numpy(), torch.cat(etiquettes).numpy()


def compresse(
  dossier_prepare: str,
  chemin_modele: str,
  taux_apprentissage: float,
  parcimonie: float,
  quantification: str,
  chemin_onnx: str,
) -> None:
  # Votre code ici
  ...

### Conseils supplémentaires

Les exports et les quantifications écrivent des fichiers intermédiaires (le modèle flottant, le modèle élagué, celui que prépare `quant_pre_process`) : un dossier temporaire les garde, le temps de la fonction. L'élagage est celui des diapositives, avec `amount=parcimonie`. `quantize_dynamic` et `quantize_static` lisent le modèle préparé par `quant_pre_process`, celui-ci avec `Calibration([image[None] for image in lot])` et `per_channel=True`, `lot` étant le premier lot de `chargeur_ent`. `rassemble_images(chargeur_test)` renvoie les images et les étiquettes que `mesure` attend.

```python
def compresse(
  dossier_prepare: str,
  chemin_modele: str,
  taux_apprentissage: float,
  parcimonie: float,
  quantification: str,
  chemin_onnx: str,
) -> None:
  torch.manual_seed(0)
  chargeur_ent = cree_chargeur(
    f"{dossier_prepare}/entrainement", TAILLE_IMAGE, melange=True
  )
  images, etiquettes = rassemble_images(
    cree_chargeur(f"{dossier_prepare}/test", TAILLE_IMAGE)
  )
  modele = LeNet()
  modele.load_state_dict(...)
  with tempfile.TemporaryDirectory() as dossier_tmp:
    chemin_flottant = f"{dossier_tmp}/flottant.onnx"
    exporte_onnx(modele, chemin_flottant)
    if parcimonie > 0:
      ...  # élaguer, réentraîner une époque, rendre l'élagage définitif
    chemin_elague = f"{dossier_tmp}/elague.onnx"
    exporte_onnx(modele, chemin_elague)
    if quantification == "aucune":
      shutil.copyfile(...)
    else:
      chemin_pretraite = f"{dossier_tmp}/pretraite.onnx"
      quant_pre_process(...)
      if quantification == "dynamique":
        quantize_dynamic(...)
      else:
        lot, _ = next(iter(chargeur_ent))
        quantize_static(...)
    metriques = {"flottant": mesure(...), "compresse": mesure(...)}
  CHEMIN_METRIQUES.write_text(json.dumps(metriques, indent=2))
```

### Solution

In [ ]:
%%writefile src/compression_modele/compression.py
import gzip
import json
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from onnxruntime.quantization import (
  CalibrationDataReader,
  QuantType,
  quantize_dynamic,
  quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader

from .donnees import TAILLE_IMAGE, cree_chargeur
from .entrainement import entraine_epoque
from .modele import LeNet

CHEMIN_METRIQUES = Path("metriques/compression.json")


def exporte_onnx(modele: nn.Module, chemin: str) -> None:
  exemple = (torch.rand(1, 3, *TAILLE_IMAGE),)
  torch.onnx.export(
    modele.eval(), exemple, chemin, dynamo=True, external_data=False, verbose=False
  )


class Calibration(CalibrationDataReader):
  def __init__(self, images: list[torch.Tensor]) -> None:
    self.images = iter(images)

  def get_next(self) -> dict | None:
    image = next(self.images, None)
    return None if image is None else {"x": image.numpy()}


def mesure(chemin: str, images: np.ndarray, etiquettes: np.ndarray) -> dict:
  session = ort.InferenceSession(chemin, providers=["CPUExecutionProvider"])
  predictions, durees = [], []
  for image in images:
    debut = time.perf_counter()
    (logits,) = session.run(None, {"x": image[None]})
    durees.append(time.perf_counter() - debut)
    predictions.append(logits.argmax())
  return {
    "exactitude": float(np.mean(np.array(predictions) == etiquettes)),
    "taille_mo": Path(chemin).stat().st_size / 1e6,
    "gzip_mo": len(gzip.compress(Path(chemin).read_bytes())) / 1e6,
    "latence_ms": float(1000 * np.median(durees[10:])),
    "p95_ms": float(1000 * np.percentile(durees[10:], 95)),
  }


def rassemble_images(chargeur: DataLoader) -> tuple[np.ndarray, np.ndarray]:
  images, etiquettes = zip(*chargeur, strict=True)
  return torch.cat(images).numpy(), torch.cat(etiquettes).numpy()


def compresse(
  dossier_prepare: str,
  chemin_modele: str,
  taux_apprentissage: float,
  parcimonie: float,
  quantification: str,
  chemin_onnx: str,
) -> None:
  torch.manual_seed(0)
  chargeur_ent = cree_chargeur(
    f"{dossier_prepare}/entrainement", TAILLE_IMAGE, melange=True
  )
  images, etiquettes = rassemble_images(
    cree_chargeur(f"{dossier_prepare}/test", TAILLE_IMAGE)
  )
  modele = LeNet()
  modele.load_state_dict(torch.load(chemin_modele))
  with tempfile.TemporaryDirectory() as dossier_tmp:
    chemin_flottant = f"{dossier_tmp}/flottant.onnx"
    exporte_onnx(modele, chemin_flottant)
    if parcimonie > 0:
      types_couches = nn.Conv2d | nn.Linear
      poids = [(m, "weight") for m in modele.modules() if isinstance(m, types_couches)]
      prune.global_unstructured(
        poids, pruning_method=prune.L1Unstructured, amount=parcimonie
      )
      optimiseur = torch.optim.AdamW(modele.parameters(), lr=taux_apprentissage)
      entraine_epoque(modele, chargeur_ent, optimiseur)
      for module, nom in poids:
        prune.remove(module, nom)
    chemin_elague = f"{dossier_tmp}/elague.onnx"
    exporte_onnx(modele, chemin_elague)
    if quantification == "aucune":
      shutil.copyfile(chemin_elague, chemin_onnx)
    else:
      chemin_pretraite = f"{dossier_tmp}/pretraite.onnx"
      quant_pre_process(chemin_elague, chemin_pretraite)
      if quantification == "dynamique":
        quantize_dynamic(chemin_pretraite, chemin_onnx, weight_type=QuantType.QInt8)
      else:
        lot, _ = next(iter(chargeur_ent))
        quantize_static(
          chemin_pretraite,
          chemin_onnx,
          Calibration([image[None] for image in lot]),
          per_channel=True,
        )
    metriques = {
      "flottant": mesure(chemin_flottant, images, etiquettes),
      "compresse": mesure(chemin_onnx, images, etiquettes),
    }
  CHEMIN_METRIQUES.write_text(json.dumps(metriques, indent=2))

## L'étape de compression

Ajoutez à `params.yaml` `parcimonie` (`0.5`), `quantification` (`statique`) et `chemin_onnx` (`classifieur_paysages.onnx`), et à la chaîne l'étape `compresse`, qui lance `python -m compression_modele compresse`, avec ses paramètres, ses dépendances, sa sortie et sa métrique, `metriques/compression.json`. Exécutez la chaîne, comparez avec `dvc metrics show` le modèle flottant et le modèle réduit, puis enregistrez le tout dans git. Le modèle réduit est-il plus petit ? Plus rapide ? Aussi exact ?

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

### Conseils supplémentaires

Les paramètres de `compresse` sont les trois nouveaux, plus `dossier_prepare`, `chemin_modele` et `taux_apprentissage`, que la fonction lit aussi. Ses dépendances sont `preparees`, le modèle entraîné et le code qu'elle exécute : `__main__.py`, `compression.py`, et les modules qu'il importe.

```
%%writefile -a params.yaml
parcimonie: ...
quantification: ...
chemin_onnx: ...
```

```python
!uv run dvc stage add -n compresse -p ... -d ... -o ... -M ... python -m compression_modele compresse
!uv run dvc repro
!uv run dvc metrics show
!git add ...
!git commit -m "..."
```

### Solution

In [ ]:
%%writefile -a params.yaml
parcimonie: 0.5
quantification: statique
chemin_onnx: classifieur_paysages.onnx

In [ ]:
!uv run dvc stage add -n compresse -p dossier_prepare,chemin_modele,taux_apprentissage,parcimonie,quantification,chemin_onnx -d preparees -d classifieur_paysages.pt -d src/compression_modele/__main__.py -d src/compression_modele/compression.py -d src/compression_modele/donnees.py -d src/compression_modele/modele.py -d src/compression_modele/entrainement.py -o classifieur_paysages.onnx -M metriques/compression.json python -m compression_modele compresse

In [ ]:
!uv run dvc repro

In [ ]:
!uv run dvc metrics show

In [ ]:
!git add .gitignore dvc.lock dvc.yaml metriques params.yaml src

In [ ]:
!git commit -m "Ajout de l'étape de compression"

Sur nos essais, le modèle élagué de moitié puis quantifié en int8 est trois fois plus petit (0,27 → 0,09 Mo), à une exactitude voisine (0,605 → 0,624 à notre dernière exécution ; à un ou deux points près, dans un sens ou dans l'autre : l'époque de réentraînement, que le modèle flottant de référence n'a pas eue, compense l'élagage, et peut même faire mieux)… mais plus lent : 0,7 ms par image au lieu de 0,45, en médiane ; le 95ᵉ centile (`p95_ms`) s'en écarte d'autant plus que la machine est occupée par ailleurs. Ce surcoût ne vient pas des conversions du format QDQ : à l'ouverture de la session, ONNX Runtime fond le graphe quantifié en opérateurs int8 (`QLinearConv` pour les convolutions, `QGemm` pour les couches linéaires), avec une seule quantification à l'entrée et une seule déquantification à la sortie. Sur la seule machine où nous l'avons mesuré, il vient des noyaux int8 eux-mêmes, lents sur un CPU sans instructions int8 rapides (VNNI), et des changements de disposition des données qu'ils demandent, que rien ne compense sur un modèle aussi petit. Un gain annoncé n'est pas un gain mesuré : mesurez sur le matériel cible.

### Expériences de compression

Lancez deux expériences : une quantification dynamique (`quantification` à `dynamique`), et un élagage de 90 % sans quantification (`parcimonie` à `0.9`, `quantification` à `aucune`). Comparez-les avec `dvc exp show`. Quel réglage donne le plus petit fichier ? Le modèle le plus rapide ? L'élagage seul réduit-il la taille du fichier ? Et celle du fichier compressé ?

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

#### Conseils supplémentaires

`-S` se répète pour changer plusieurs paramètres. Seule `compresse` s'exécute.

```python
!uv run dvc exp run -S ...
!uv run dvc exp run -S ... -S ...
!uv run dvc exp show --only-changed
```

#### Solution

In [ ]:
!uv run dvc exp run -S quantification=dynamique

In [ ]:
!uv run dvc exp run -S parcimonie=0.9 -S quantification=aucune

In [ ]:
!uv run dvc exp show --only-changed

Les deux quantifications donnent des fichiers trois fois plus petits, et la dynamique est la plus lente (2,5 ms par image sur nos essais) : elle calcule les plages des activations à chaque inférence, et les convolutions en entiers d'ONNX Runtime (`ConvInteger`) sont lentes sur CPU. L'élagage seul ne réduit pas la taille du fichier : les zéros y sont stockés comme les autres poids, et les calculs ne vont pas plus vite. Mais le fichier compressé est cinq fois plus petit (0,23 → 0,05 Mo) : les zéros se compressent bien, ce qui compte pour télécharger un modèle. Et l'exactitude baisse à peine (0,605 → 0,600) : l'époque de réentraînement compense l'élagage, signe que ce LeNet est surdimensionné pour ces images.